This is the endpoint for experiment 2 - a barebones model that retrieves data from lakebase using the same structure as the original Fraud Model

In [0]:
%pip install databricks-sdk --upgrade
%restart_python

In [0]:
CLIENT_ID = dbutils.secrets.get('shm', 'sp-id')
CLIENT_SECRET = dbutils.secrets.get('shm', 'sp-auth')
OIDC_TOKEN_URL = "https://adb-984752964297111.11.azuredatabricks.net/oidc/v1/token"

In [0]:
import requests

data = {
    "grant_type": "client_credentials",
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "scope": "all-apis",
}
headers = {"Content-Type": "application/x-www-form-urlencoded"}

resp = requests.post(OIDC_TOKEN_URL, data=data, headers=headers, timeout=10)
resp.raise_for_status()
token = resp.json()["access_token"]

In [0]:

import os
UC_MODEL_NAME = "cjc.default.lakebase_adder_model"
ENDPOINT_NAME = "shm_lakebase_adder_endpoint"
PG_INSTANCE_NAME = "dbdemo-travel-online-store"
os.environ["CLIENT_ID"] = dbutils.secrets.get('shm', 'sp-id')
os.environ["CLIENT_SECRET"] = dbutils.secrets.get('shm', 'sp-auth')

In [0]:
from typing import Dict, Any
import pandas as pd
import mlflow
import psycopg2
from databricks.sdk import WorkspaceClient
import uuid

class LakebaseAdderModel(mlflow.pyfunc.PythonModel):
    """
    Blank pyfunc to test latency
    """
    def __init__(self):
        # Put any initialization logic here (load artifacts, etc.)
        self.db_read_write_dns = "instance-cadddfbf-f85a-45a4-a558-cac5abf1b659.database.azuredatabricks.net"
        self.db_name = "scott.mckean@databricks.com"
        self.pg_table = "random_numbers"
        self.pg_instance =  "dbdemo-travel-online-store"

    def load_context(self, context):
        import os
        import requests
        self.client_id = os.environ["CLIENT_ID"]
        self.client_secret = os.environ["CLIENT_SECRET"]

        data = {
            "grant_type": "client_credentials",
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "scope": "all-apis",
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}

        resp = requests.post(OIDC_TOKEN_URL, data=data, headers=headers, timeout=10)
        resp.raise_for_status()
        self.oauth_token = resp.json()["access_token"]

    def predict(self, model_input: pd.DataFrame) -> Dict[str, Any]:
        # setup PG with token each request
        conn = psycopg2.connect(
            host=self.db_read_write_dns,
            dbname="databricks_postgres",
            user=self.client_id,
            password=self.oauth_token,            
            sslmode="require",
            connect_timeout=10,
        )

        pg_df = df = pd.read_sql(f"""
            SELECT 
            AVG(col1) as col1, 
            AVG(col2) as col2, 
            AVG(col3) as col3 
            FROM {self.pg_table} 
            LIMIT {model_input.shape[0]};
            """, conn)
        
        print(pg_df)

        sum_result = (
          model_input.iloc[:, 0]
          + model_input.iloc[:, 1]
          + model_input.iloc[:, 2]
        ).astype(float)

        sum_result = sum_result.add(pg_df.sum(axis=1), fill_value=0)

        conn.close()

        # Return in Databricks Model Serving tabular format
        return sum_result

In [0]:
df = pd.DataFrame(
    data=[(1., 2., 3.), (4., 5., 6.), (7., 8., 9.)],
    columns=['col1', 'col2', 'col3']
)
display(df)

In [0]:
model = LakebaseAdderModel()
model.load_context(None)
model.predict(df)

In [0]:
# Ship as a model and serve as a CPU endpoint
with mlflow.start_run():
    model = LakebaseAdderModel()
    
    model_info = mlflow.pyfunc.log_model(
      artifact_path="model",
      python_model=model,
      registered_model_name=UC_MODEL_NAME,
      input_example=df,
      pip_requirements=[
        "mlflow",
        "psycopg2-binary",
        "requests",
        "pandas",
    ],
    )

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ServedModelInput,
    EndpointCoreConfigInput,
    ServedModelInputWorkloadType,
)

w = WorkspaceClient()

served_model = ServedModelInput(
    model_name=UC_MODEL_NAME,
    model_version=model_info.registered_model_version,
    workload_type=ServedModelInputWorkloadType.CPU,
    workload_size="Small",
    environment_vars={
        "CLIENT_ID": "{{secrets/shm/sp-id}}",
        "CLIENT_SECRET": "{{secrets/shm/sp-auth}}",
    },
    scale_to_zero_enabled=True
)

config = EndpointCoreConfigInput(
    name=ENDPOINT_NAME, 
    served_models=[served_model]
    )

In [0]:
try:
    w.serving_endpoints.update_config(
        name=ENDPOINT_NAME,
        served_models=[served_model],
    ).result()
    print(f"✅ Endpoint '{ENDPOINT_NAME}' updated")
except Exception:
    w.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=config,
    ).result()
    print(f"✅ Endpoint '{ENDPOINT_NAME}' created")